# SETUP

In [ ]:
import itertools
import math
import numpy as np
import os
import pandas as pd
import skimage.io

from datetime import datetime
from openpyxl import load_workbook
from pathlib import Path
from scipy.ndimage import gaussian_filter

import resource

## INPUT

In [ ]:
dir_input = "INPUT FOLDER"

### PREFERENCES

In [ ]:
distance_unit = "µm"
dist_per_px = 1 / 1
quantification_bins = 6
include_control_ratios = False
control_channels = ["DAPI"]
output_raw_values = False

## INPUT CHECK

In [ ]:
if type(dir_input) != str:
    raise TypeError("dir_input must be a string.")

if type(distance_unit) != str:
    print("distance_unit must be a string; using default value (µm).")

#dist_per_px
try:
    dist_per_px = float(dist_per_px)
    if not dist_per_px > 0:
        print("dist_per_px must be greater than 0; using default value (1). Distance units = pixels.")
except:
    dist_per_px = 1
    print("dist_per_px must be greater than 0; using default value (1). Distance units = pixels.")

#quantification_bins
try:
    quantification_bins_f = float(quantification_bins)
    quantification_bins = float(round(quantification_bins))
    if quantification_bins < 1:
        quantification_bins = 6
        print("quantification_bins must be 1 or greater; using default value (6).")
    elif quantification_bins == 1:
        print("Caution: are you sure you want to segment data using a single \"bin\"?")
    elif quantification_bins != quantification_bins_f:
        print(f'quantification_bins was rounded from {quantification_bins_f} to {quantification_bins}')
    quantification_bins = int(quantification_bins)
except:
    quantification_bins = 6
    print("quantification_bins must be 1 or greater; using default value (6).")

#control_channels
control_channels_f = [x for x in control_channels if type(x) == str]
if len(control_channels_f) < len(control_channels):
    print(f'control_channels must be strings; {len(control_channels) - len(control_channels_f)} elements were removed from control_channels.')
control_channels = control_channels_f

#include_control_ratios
if type(include_control_ratios) != bool:
    include_control_ratios = False
    print("include_control_ratios must be True or False; using default value (False).")

#output_raw_values
if type(output_raw_values) != bool:
    output_raw_values = False
    print("output_raw_values must be True or False; using default value (False).")

In [ ]:
bin_fraction = 1 / quantification_bins
bin_index = [f'Bin {x + 1}' for x in range(quantification_bins)] + ["Overall"]
now = datetime.now() 
dt_string = now.strftime("%Y-%m-%d_%H%M%S")

# FUNCTIONS

In [ ]:
def filter_hidden(some_list):
    new_list = sorted([x for x in some_list if "._" not in x])
    if len(new_list) < len(some_list):
        print(f'Detected {len(some_list) - len(new_list)} hidden files in the input directory\n'
              + f'Hidden image files have been excluded from analysis.')
    return(new_list)

In [ ]:
def summary():
    analysis_parameters = f'Analysis parameters...\n\n'
    analysis_parameters += f'Bins (regions): {quantification_bins}\n'
    analysis_parameters += f'Common channels: {control_channels}\n'
    summary = open(f'{outputdir}Analysis parameters for {dir_input}.txt', "w")
    summary.write(analysis_parameters)
    summary.close()

In [ ]:
def tidy_columns(df_raw):
    df = df_raw.copy()
    for column in df.columns:
        if column in ["ImageNumber",
                      "ObjectNumber",
                      "Metadata_Channel",
                      "Metadata_Frame",
                      "Metadata_Processing",
                      "Metadata_Series"]:
            df.drop([column], axis = 1, inplace = True)
    df.columns = [x.split("Metadata_")[-1] for x in df.columns]
    df['Distance_Minimum_Spheroid'] = (1 - (df['Distance_Minimum_Spheroid'] / r_max)).clip(lower = 0.0001)
    df = df.rename(columns = {'Intensity_IntegratedIntensity_Masked_Target': f'Intensity_IntegratedIntensity_Masked_{target}',
                                              'Distance_Minimum_Spheroid': 'Frac_Dist_Core'})
    intensity_cols = [x for x in df.columns if "Intensity_IntegratedIntensity_Masked_" in x]
    image_channels = [x.split("_")[-1] for x in intensity_cols]
    df = df.rename(columns = dict(zip(intensity_cols, image_channels)))
    return(df, image_channels)

In [ ]:
def try_dir(some_dir):
    if not os.path.exists(some_dir):
        os.makedirs(some_dir)

# FILE PARSE

In [ ]:
path_files = sorted([str(path_file) for path_file in Path(dir_input).rglob("*.csv")])
files = filter_hidden(path_files)
file_details = [Path(file_name).stem.split("_") for file_name in path_files]
celltypes = sorted(set([details[0] for details in file_details]))
targets = sorted(set([details[2] for details in file_details]))
all_dfs = [pd.read_csv(df) for df in path_files]
info_and_dfs = list(zip(file_details, all_dfs))

# MAIN

In [ ]:
for celltype in celltypes:
    celltype_flist = [x for x in file_details if x[0] == celltype]
    if not celltype_flist:
        continue
    all_channels_of_interest = []
    targets = sorted(set(details[2] for details in celltype_flist))
    celltype_df_datasets = []
    for target in targets:
        target_flist = [x for x in celltype_flist if x[2] == target]
        if not target_flist:
            continue
        celltype_target_dfs = []
        replicates = sorted(set(details[1] for details in target_flist))
        for replicate in replicates:
            replicate_flist = [x for x in target_flist if x[1] == replicate]
            if not replicate_flist:
                continue
            replicate_images_dfs = []
            for image_details in replicate_flist:
                image_no = image_details[3]
                image_csv = f'{"_".join(image_details)}.csv'
                image_data = pd.read_csv(Path(dir_input, image_csv))
                r_max = image_data['Distance_Minimum_Spheroid'].max()
                working_df, image_channels = tidy_columns(image_data)
                channels_of_interest = [x for x in image_channels if x not in control_channels]
                channels = [x for x in control_channels if x in image_channels] + channels_of_interest
                channel_combinations = list(itertools.combinations(channels, r=2))
                ratio_labels = [f'{c1}/{c0}' for c0, c1 in channel_combinations]
                bin_info_headers = ["Objects", f'Mean distance {distance_unit}']
                bin_lists = []
                for channel in channels:
                    bin_info_headers.append(f'{channel}')
                for i in range(quantification_bins):
                    lower_dist_limit = bin_fraction * i
                    upper_dist_limit = bin_fraction * (i + 1)
                    bin_slice = working_df[(lower_dist_limit < working_df['Frac_Dist_Core']) & (working_df['Frac_Dist_Core'] <= upper_dist_limit)]
                    mean_dist_bin = bin_slice['Frac_Dist_Core'].mean() * r_max * dist_per_px
                    bin_data = [len(bin_slice.index), mean_dist_bin]
                    for channel in channels:
                        bin_data.append(bin_slice[f'{channel}'].sum())
                    bin_lists.append(bin_data)
                bin_df = pd.DataFrame(bin_lists, columns = bin_info_headers)
                overall = pd.Series([bin_df[x].sum() for x in bin_df.columns])
                bin_df.loc[len(bin_df.index)] = dict(zip(bin_info_headers, overall))
                bin_df.index = bin_index
                bin_df.loc["Overall", f'Mean distance {distance_unit}'] = working_df['Frac_Dist_Core'].mean() * r_max * dist_per_px
                image_summary_df = pd.DataFrame()
                for ratio_label in ratio_labels:
                    c0, c1 = ratio_label.split("/")
                    image_summary_df[f'{ratio_label}'] = bin_df[f'{c0}'] / bin_df[f'{c1}']
                image_summary_df = pd.concat([bin_df.iloc[:,:2], image_summary_df], axis = 1)
                image_summary_df['Objects'] = image_summary_df['Objects'].astype("int")
                if not include_control_ratios:
                    cols_to_drop = [x for x in image_summary_df.columns
                                    if any([y for y in control_channels if y in x])
                                    and not [z for z in channels_of_interest if z in x]]
                    image_summary_df = image_summary_df.drop(columns = cols_to_drop)
                    ratio_labels = sorted([x for x in ratio_labels if x not in cols_to_drop])
                replicate_images_dfs.append(image_summary_df.copy())
                all_channels_of_interest.extend(channels_of_interest)
            replicate_summary_df = sum(replicate_images_dfs) / len(replicate_images_dfs)
            celltype_target_dfs.append(replicate_summary_df.copy())
        celltype_target_ratio_dfs = []
        for ratio_label in ratio_labels:
            values_lists = [x[f'{ratio_label}'] for x in celltype_target_dfs]
            ratio_df = pd.DataFrame(values_lists)
            midx = pd.MultiIndex.from_arrays([[celltype] * len(replicates),
                                              [f'{ratio_label.split("/")[0]}'] * len(replicates),
                                              [f'{ratio_label.split("/")[1]}'] * len(replicates),
                                               replicates],
                                               names = ["Celltype",
                                                        "Target channel",
                                                        "Control channel",
                                                        "Replicate"])
            ratio_df.index = midx
            celltype_target_ratio_dfs.append(ratio_df) # types of ratios
        celltype_df_datasets.append((celltype_target_ratio_dfs, target)) # targets
    all_channels_of_interest = sorted(set(all_channels_of_interest))
    all_channels_of_interest = str(all_channels_of_interest).translate({ord(c): None for c in "'"})[:230]
    if len(celltypes) > 1:
        dir_output = Path("output", f'{dt_string} - {dir_input} - normalized spatial signal analysis ({quantification_bins} bins) ')
        out_name = f'{celltype}  {all_channels_of_interest}.xlsx'
    else:
        dir_output = Path("output")
        out_name = f'{dt_string} - {dir_input} - normalized spatial signal analysis ({quantification_bins} bins) {celltype} {all_channels_of_interest}.xlsx'
    try_dir(dir_output)
    blank_df = pd.DataFrame()
    with pd.ExcelWriter(Path(dir_output, out_name)) as writer:
        blank_df.to_excel(writer)
    with pd.ExcelWriter(Path(dir_output, out_name), mode = "a", engine = "openpyxl", if_sheet_exists = "overlay") as writer:
        for celltype_target_dfset in celltype_df_datasets:
            row_tracker = 0
            ratio_dfs = celltype_target_dfset[0]
            target_name = celltype_target_dfset[1]
            for ratio_df in ratio_dfs:
                if not output_raw_values:
                    ratio_df = ratio_df.T / ratio_df.T.loc['Overall',:]
                    ratio_df = (ratio_df / ratio_df.loc['Bin 1',:].mean()).T
                ratio_df.to_excel(writer, sheet_name = f'{celltype} {target_name}'[:31], startrow = row_tracker)
                row_tracker += (len(ratio_df.index) + 2)
    workbook = load_workbook(Path(dir_output, out_name))
    if "Sheet1" in workbook.sheetnames:
        workbook.remove(workbook['Sheet1'])
        workbook.save(Path(dir_output, out_name))

In [ ]:
now = datetime.now()
print ("\n**********SCRIPT COMPLETE**********")
print(now.strftime("%Y-%m-%d_%H%M%S"))